## Contiguity is the hidden killer

Let me tell a story: When we finished the development of RoPE kernel, we immediately deploy it to production training. However, the loss has obvious divergence. In the beginning, we thought our implementation was wrong, so we kept adding difficult correctneess test to it, but they all passed. We also found that if we are using "Flash-Attn" instead of "Sdpa" at the attention layer, the loss does not diverge. Weird, Right?





### Ensure you are using GPU by running `nvidia-smi`

In [ ]:
!nvidia-smi

Wed Aug 21 17:59:05 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   42C    P8               9W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

### Install dependencies

In [ ]:
!pip install liger-kernel transformers

  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinu

### Original Rope Implementation

The implementation looks solid. Right?

In [ ]:
import triton
import torch
from liger_kernel.ops.rope import _triton_rope

class LigerRopeFunction(torch.autograd.Function):

    @staticmethod
    def forward(ctx, q, k, cos, sin, position_ids=None, unsqueeze_dim=1):
        """
        q size: (bsz, n_q_head, seq_len, head_dim)
        k size: (bsz, n_kv_head, seq_len, head_dim)
        cos size: (1, seq_len, head_dim)
        sin size: (1, seq_len, head_dim)
        """

        # transpose it back to the physical shape because Triton looks at the physical storage
        # note: q and k are incontiguous before the transformation and will become contiguous after transpose
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)

        batch_size, seq_len, n_q_head, head_dim = q.shape
        n_kv_head = k.shape[2]
        pad_hd = triton.next_power_of_2(head_dim)
        pad_n_q_head = triton.next_power_of_2(n_q_head)
        pad_n_kv_head = triton.next_power_of_2(n_kv_head)
        BLOCK_SIZE = max(pad_n_q_head, pad_n_kv_head)

        n_row = batch_size * seq_len

        _triton_rope[(n_row,)](
            q,
            q.stride(1),
            k,
            k.stride(1),
            cos,
            cos.stride(-2),
            sin,
            sin.stride(-2),
            batch_size,
            seq_len,
            n_q_head,
            n_kv_head,
            head_dim,
            pad_n_q_head,
            pad_n_kv_head,
            pad_hd,
            BLOCK_SIZE=BLOCK_SIZE,
            BACKWARD_PASS=False,
        )

        ctx.save_for_backward(cos, sin)
        return q.transpose(1, 2), k.transpose(1, 2)

def liger_rotary_pos_emb(q, k, cos, sin, position_ids=None, unsqueeze_dim=1):
    return LigerRopeFunction.apply(q, k, cos, sin, position_ids, unsqueeze_dim)

## Divergence with Hugging Face implementation

In [ ]:
from transformers.models.llama.modeling_llama import (
    LlamaRotaryEmbedding,
    apply_rotary_pos_emb,
)

atol, rtol=1e-5, 1e-5
head_dim = 64
num_q_heads = 32
num_kv_heads = 8
bsz = 2
seq_len = 128
dtype = torch.float32

rotary_emb = LlamaRotaryEmbedding(head_dim, device="cuda")

q = (
    torch.randn((bsz, num_q_heads, seq_len, head_dim), device="cuda")
    .to(dtype)
)

k = (
    torch.randn((bsz, num_kv_heads, seq_len, head_dim), device="cuda")
    .to(dtype)
)

pos_ids = torch.arange(seq_len, device="cuda", dtype=torch.long).unsqueeze(0)
cos, sin = rotary_emb(k, pos_ids)

# validate forward pass
hf_q, hf_k = apply_rotary_pos_emb(q, k, cos, sin, pos_ids)
tt_q, tt_k = liger_rotary_pos_emb(q, k, cos, sin)
assert torch.allclose(hf_q, tt_q, atol=atol, rtol=rtol)
assert torch.allclose(hf_k, tt_k, atol=atol, rtol=rtol)

AssertionError: 

### Fix the contiguity

In turned out it was because the tensor from Sdpa is not contiguous, but the one from flash-attn is. And our rope kernel does not enforce the contiguity! After adding it, we are able be exact

In [ ]:
import triton
import torch

class LigerRopeFunction(torch.autograd.Function):

    @staticmethod
    def forward(ctx, q, k, cos, sin, position_ids=None, unsqueeze_dim=1):
        """
        q size: (bsz, n_q_head, seq_len, head_dim)
        k size: (bsz, n_kv_head, seq_len, head_dim)
        cos size: (1, seq_len, head_dim)
        sin size: (1, seq_len, head_dim)
        """

        # transpose it back to the physical shape because Triton looks at the physical storage
        # note: q and k are incontiguous before the transformation and will become contiguous after transpose
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)

        batch_size, seq_len, n_q_head, head_dim = q.shape
        n_kv_head = k.shape[2]
        pad_hd = triton.next_power_of_2(head_dim)
        pad_n_q_head = triton.next_power_of_2(n_q_head)
        pad_n_kv_head = triton.next_power_of_2(n_kv_head)
        BLOCK_SIZE = max(pad_n_q_head, pad_n_kv_head)

        n_row = batch_size * seq_len

        # Added: enforce the tensor to be contiguous
        q = q.contiguous()
        k = k.contiguous()

        _triton_rope[(n_row,)](
            q,
            q.stride(1),
            k,
            k.stride(1),
            cos,
            cos.stride(-2),
            sin,
            sin.stride(-2),
            batch_size,
            seq_len,
            n_q_head,
            n_kv_head,
            head_dim,
            pad_n_q_head,
            pad_n_kv_head,
            pad_hd,
            BLOCK_SIZE=BLOCK_SIZE,
            BACKWARD_PASS=False,
        )

        ctx.save_for_backward(cos, sin)
        return q.transpose(1, 2), k.transpose(1, 2)

def liger_rotary_pos_emb(q, k, cos, sin, position_ids=None, unsqueeze_dim=1):
    return LigerRopeFunction.apply(q, k, cos, sin, position_ids, unsqueeze_dim)


### We can pass the test after the fix

In [ ]:
atol, rtol=1e-5, 1e-5
head_dim = 64
num_q_heads = 32
num_kv_heads = 8
bsz = 2
seq_len = 128
dtype = torch.float32

rotary_emb = LlamaRotaryEmbedding(head_dim, device="cuda")

q = (
    torch.randn((bsz, num_q_heads, seq_len, head_dim), device="cuda")
    .to(dtype)
)

k = (
    torch.randn((bsz, num_kv_heads, seq_len, head_dim), device="cuda")
    .to(dtype)
)

pos_ids = torch.arange(seq_len, device="cuda", dtype=torch.long).unsqueeze(0)
cos, sin = rotary_emb(k, pos_ids)

# validate forward pass
hf_q, hf_k = apply_rotary_pos_emb(q, k, cos, sin, pos_ids)
tt_q, tt_k = liger_rotary_pos_emb(q, k, cos, sin)
assert torch.allclose(hf_q, tt_q, atol=atol, rtol=rtol)
assert torch.allclose(hf_k, tt_k, atol=atol, rtol=rtol)